In [10]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

Device: cpu
PyTorch: 2.10.0+cpu


In [11]:
# CIFAR-10 mean/std are dataset-specific constants — using ImageNet values 
# here would be technically wrong, even though the difference is small.
# These come from computing stats over the actual CIFAR-10 training set.
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2023, 0.1994, 0.2010)

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4), # standard augmentation for CIFAR
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD)
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD)
])

trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform_train
)
testset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform_test
)

# 2000 test samples is enough for CKA and Grad-CAM analysis.
# Running on the full 10k test set would take much longer with no meaningful 
# difference in the findings.
test_subset = torch.utils.data.Subset(testset, range(2000))

trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=128, shuffle=True, num_workers=2
)
testloader = torch.utils.data.DataLoader(
    test_subset, batch_size=64, shuffle=False, num_workers=2
)

CLASSES = ['airplane','automobile','bird','cat','deer',
           'dog','frog','horse','ship','truck']

print(f"Train: {len(trainset):,} samples | Test subset: {len(test_subset):,} samples")
print(f"Input shape check:")
images, labels = next(iter(testloader))
print(f"  Batch shape: {images.shape}")

Train: 50,000 samples | Test subset: 2,000 samples
Input shape check:
  Batch shape: torch.Size([64, 3, 32, 32])


In [12]:
model = models.resnet18(weights=None)  # train from scratch

# Adapt for 32x32 CIFAR-10 input.
# Original: 7x7 conv, stride 2 → aggressively downsamples 224x224 images.
# For 32x32, this destroys spatial information immediately.
# Fix: 3x3 conv, stride 1, no maxpool - standard for CIFAR in the literature.
model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
model.maxpool = nn.Identity()
model.fc = nn.Linear(512, 10)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
size_mb = total_params * 4 / 1024**2

print(f"Parameters: {total_params:,}")
print(f"Model size (float32): {size_mb:.1f} MB")

with torch.no_grad():
    test_input = torch.randn(1, 3, 32, 32).to(device)
    test_output = model(test_input)
    print(f"Output shape: {test_output.shape}")

Parameters: 11,173,962
Model size (float32): 42.6 MB
Output shape: torch.Size([1, 10])


In [ ]:
def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

# With random initialisation, accuracy should be ~10% (random chance for 10 classes).
acc_random = evaluate(model, testloader, device)
print(f"Accuracy at random init (expected ~10%): {acc_random:.2f}%")

Accuracy at random init (expect ~10%): 10.25%


In [ ]:
criterion = nn.CrossEntropyLoss()

# SGD (stochastic gradient descent) with higher initial LR for training from scratch.
# Pretrained fine-tuning uses lr=0.01 to avoid destroying learned weights.
# Scratch training uses lr=0.1 (standard in He et al. and compression papers)
optimizer = torch.optim.SGD(
    model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4
)

# MultiStepLR: drop LR by 10x at epoch 100 and 150.
# For our 15-epoch run, we do a single drop at epoch 10.
scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer, milestones=[10, 13], gamma=0.1
)

os.makedirs('checkpoints', exist_ok=True)

best_acc = 0.0
EPOCHS = 15

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for images, labels in tqdm(trainloader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    scheduler.step()
    val_acc = evaluate(model, testloader, device)
    avg_loss = running_loss / len(trainloader)
    current_lr = optimizer.param_groups[0]['lr']

    print(f"Epoch {epoch+1:2d} | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.2f}% | LR: {current_lr:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'checkpoints/resnet18_cifar10_best.pth')
        print(f"Saved ({best_acc:.2f}%)")

print(f"\nBest accuracy: {best_acc:.2f}%")

Epoch 1/15:  19%|█▊        | 73/391 [05:34<24:15,  4.58s/it] 


KeyboardInterrupt: 

In [15]:
import json

model.load_state_dict(
    torch.load('checkpoints/resnet18_cifar10_best.pth', map_location=device)
)
final_acc = evaluate(model, testloader, device)

baseline_stats = {
    "model": "ResNet-18 (CIFAR-adapted)",
    "dataset": "CIFAR-10 (2000 test subset)",
    "total_parameters": total_params,
    "size_float32_mb": round(size_mb, 2),
    "accuracy_random_init": round(acc_random, 2),
    "accuracy_after_finetuning": round(final_acc, 2),
    "epochs_trained": EPOCHS,
    "notes": "Conv1 replaced with 3x3 stride-1, maxpool removed for 32x32 input"
}

os.makedirs('results/tables', exist_ok=True)
with open('results/tables/baseline_stats.json', 'w') as f:
    json.dump(baseline_stats, f, indent=2)

print(json.dumps(baseline_stats, indent=2))

{
  "model": "ResNet-18 (CIFAR-adapted)",
  "dataset": "CIFAR-10 (2000 test subset)",
  "total_parameters": 11173962,
  "size_float32_mb": 42.63,
  "accuracy_random_init": 10.25,
  "accuracy_after_finetuning": 90.15,
  "epochs_trained": 15,
  "notes": "Conv1 replaced with 3x3 stride-1, maxpool removed for 32x32 input"
}
